In [218]:
import math
import random
import numpy
import torch
import torch.nn as nn
from torch.nn import functional as F

In [219]:
batch_size = 64
block_size = 256
n_embd = 256
n_head = 4
epochs = 10000
learning_rate = 3e-4
n_layers = 6
dropout = 0.2
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"using device: {device}")

using device: mps


In [220]:
with open('data/grabbed_nietzsche.txt', 'r', encoding='utf-8') as f:
    nietzshe = f.read()

In [221]:
len(nietzshe)

5812740

In [222]:
characters = sorted(set(nietzshe))
vocab_size = len(characters)
print(''.join(characters))
print(vocab_size)


 !"&'()*,-./0123456789:;<=>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]_abcdefghijklmnopqrstuvwxyz}§ÀÄÆÈÉÊÏÜàâäæçèéêîïôöùûüŒœΚΣΤάέήίαβγδεζηθικλμνξοπρςστυφχψωόύώἀἃἄἆἈἐἑἒἔἰὀὖὰὲὶὸᾶ᾽ῑῡῢῶ‐–—‘’“”…
177


In [223]:
stoi = {s:i for i, s in enumerate(characters)}
itos = {i:s for s, i in stoi.items()}
print(stoi)
print(itos)                                     # the encoders we are using are character to token encoders

encode = lambda s: [stoi[c] for c in s]
decode = lambda s: ''.join([itos[c] for c in s])
encode('hi there')
decode(encode("hi there"))


{'\n': 0, ' ': 1, '!': 2, '"': 3, '&': 4, "'": 5, '(': 6, ')': 7, '*': 8, ',': 9, '-': 10, '.': 11, '/': 12, '0': 13, '1': 14, '2': 15, '3': 16, '4': 17, '5': 18, '6': 19, '7': 20, '8': 21, '9': 22, ':': 23, ';': 24, '<': 25, '=': 26, '>': 27, '?': 28, 'A': 29, 'B': 30, 'C': 31, 'D': 32, 'E': 33, 'F': 34, 'G': 35, 'H': 36, 'I': 37, 'J': 38, 'K': 39, 'L': 40, 'M': 41, 'N': 42, 'O': 43, 'P': 44, 'Q': 45, 'R': 46, 'S': 47, 'T': 48, 'U': 49, 'V': 50, 'W': 51, 'X': 52, 'Y': 53, 'Z': 54, '[': 55, '\\': 56, ']': 57, '_': 58, 'a': 59, 'b': 60, 'c': 61, 'd': 62, 'e': 63, 'f': 64, 'g': 65, 'h': 66, 'i': 67, 'j': 68, 'k': 69, 'l': 70, 'm': 71, 'n': 72, 'o': 73, 'p': 74, 'q': 75, 'r': 76, 's': 77, 't': 78, 'u': 79, 'v': 80, 'w': 81, 'x': 82, 'y': 83, 'z': 84, '}': 85, '§': 86, 'À': 87, 'Ä': 88, 'Æ': 89, 'È': 90, 'É': 91, 'Ê': 92, 'Ï': 93, 'Ü': 94, 'à': 95, 'â': 96, 'ä': 97, 'æ': 98, 'ç': 99, 'è': 100, 'é': 101, 'ê': 102, 'î': 103, 'ï': 104, 'ô': 105, 'ö': 106, 'ù': 107, 'û': 108, 'ü': 109, 'Œ': 11

'hi there'

In [224]:
tokenized_nietzshe = torch.tensor(encode(nietzshe), dtype=torch.long)       # encode the entirety of nietzshe into tokens
                                                                            

tokenized_nietzshe[:1000]

tensor([ 48,  36,  33,   1,  48,  51,  37,  40,  37,  35,  36,  48,   1,  43,
         34,   1,  48,  36,  33,   1,  37,  32,  43,  40,  47,   0,   0,  30,
         53,   0,   0,  34,  46,  37,  33,  32,  46,  37,  31,  36,   1,  42,
         37,  33,  48,  54,  47,  31,  36,  33,   0,   0,  43,  76,   9,   1,
         36,  73,  81,   1,  78,  73,   1,  44,  66,  67,  70,  73,  77,  73,
         74,  66,  67,  77,  63,   1,  81,  67,  78,  66,   1,  78,  66,  63,
          1,  36,  59,  71,  71,  63,  76,   0,   0,   0,  48,  36,  33,   1,
         29,  42,  48,  37,  31,  36,  46,  37,  47,  48,   0,   0,  58,  42,
         43,  48,  33,  47,   1,  48,  43,   1,  54,  29,  46,  29,  48,  36,
         49,  47,  48,  46,  29,   9,   1,  29,  42,  32,   1,  33,  48,  33,
         46,  42,  29,  40,   1,  46,  33,  31,  49,  46,  46,  33,  42,  31,
         33,  58,   0,   0,   0,  48,  46,  29,  42,  47,  40,  29,  48,  33,
         32,   1,  30,  53,   0,   0,  29,  42,  48,  36,  43,  

In [225]:
# split into train and test data
n = int(len(tokenized_nietzshe)*.9)
train_nietzshe = tokenized_nietzshe[:n]
test_nietzshe = tokenized_nietzshe[n:]

print(train_nietzshe.shape, test_nietzshe.shape)

torch.Size([5231466]) torch.Size([581274])


In [226]:
train_nietzshe[:block_size+1]

tensor([48, 36, 33,  1, 48, 51, 37, 40, 37, 35, 36, 48,  1, 43, 34,  1, 48, 36,
        33,  1, 37, 32, 43, 40, 47,  0,  0, 30, 53,  0,  0, 34, 46, 37, 33, 32,
        46, 37, 31, 36,  1, 42, 37, 33, 48, 54, 47, 31, 36, 33,  0,  0, 43, 76,
         9,  1, 36, 73, 81,  1, 78, 73,  1, 44, 66, 67, 70, 73, 77, 73, 74, 66,
        67, 77, 63,  1, 81, 67, 78, 66,  1, 78, 66, 63,  1, 36, 59, 71, 71, 63,
        76,  0,  0,  0, 48, 36, 33,  1, 29, 42, 48, 37, 31, 36, 46, 37, 47, 48,
         0,  0, 58, 42, 43, 48, 33, 47,  1, 48, 43,  1, 54, 29, 46, 29, 48, 36,
        49, 47, 48, 46, 29,  9,  1, 29, 42, 32,  1, 33, 48, 33, 46, 42, 29, 40,
         1, 46, 33, 31, 49, 46, 46, 33, 42, 31, 33, 58,  0,  0,  0, 48, 46, 29,
        42, 47, 40, 29, 48, 33, 32,  1, 30, 53,  0,  0, 29, 42, 48, 36, 43, 42,
        53,  1, 41, 11,  1, 40, 49, 32, 43, 50, 37, 31, 37,  0,  0,  0, 48, 66,
        63,  1, 31, 73, 71, 74, 70, 63, 78, 63,  1, 51, 73, 76, 69, 77,  1, 73,
        64,  1, 34, 76, 67, 63, 62, 76, 

In [227]:
v = torch.randint(n-block_size, (batch_size, ))     # how to sample random values from the trainset
for i in v:
    print(i.item())

4428702
941982
1653666
1041271
4266791
555075
4471304
4864967
1839411
3084018
1663499
2351286
3393602
2929787
2465281
1216579
3612818
1727690
2248491
3545406
3599406
1593920
3874838
4086673
4544356
1147609
4168627
522805
1959926
1963690
2055703
1080474
3243446
2597928
2543501
5040615
683450
4784453
118430
2937437
1717923
3565788
3615144
4480294
1844393
434288
300492
2240045
4115046
542692
5008406
3577446
886539
547062
274370
1773248
3827697
3853538
5203349
490972
2917553
2657900
1880948
4219602


In [228]:

test_nietzshe = test_nietzshe.to(device)
train_nietzshe = train_nietzshe.to(device)

def generate_batch(type=None):
    xb = []
    yb = []

    if type == 'test':
        data = test_nietzshe
    else:
        data = train_nietzshe

    start_vals = torch.randint(len(data)-block_size, (batch_size, ), device=device)
    offsets = torch.arange(block_size, device=device)
    xb = data[start_vals[:, None] + offsets]
    yb = data[start_vals[:, None] + offsets + 1]
    # n = len(data)
    # start_vals = torch.randint(n-block_size, (batch_size, ))     # how to sample random values from the trainset 
    # for start_val in start_vals:
    #     start_val = start_val.item()
    #     xb.append(data[start_val:start_val+block_size])
    #     yb.append(data[start_val+1:start_val+block_size+1])
    
    # xb = torch.stack(xb)
    # yb = torch.stack(yb)
    #return xb.to(device), yb.to(device)                          # move batches onto the active device

    return xb, yb

xb, yb = generate_batch()

print(xb, yb)

tensor([[ 71,  59,  65,  ...,  73,  81,   1],
        [ 63,  62,   1,  ...,  63,  76,  77],
        [ 77,   1,  77,  ...,  78,  66,  63],
        ...,
        [ 76,   1,  77,  ...,  78,  76,  79],
        [ 73,  72,  77,  ...,  11,  58, 171],
        [ 62,   1,  59,  ...,  73,  80,  63]], device='mps:0') tensor([[ 59,  65,  63,  ...,  81,   1,  81],
        [ 62,   1,  78,  ...,  76,  77,   1],
        [  1,  77,  78,  ...,  66,  63,  67],
        ...,
        [  1,  77,  78,  ...,  76,  79,  67],
        [ 72,  77,   1,  ...,  58, 171,  48],
        [  1,  59,  76,  ...,  80,  63,  76]], device='mps:0')


In [229]:
class SelfAttention(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size)
        self.query = nn.Linear(n_embd, head_size)
        self.value = nn.Linear(n_embd, head_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, xb):                                              # (B, T, C)
        k = self.key(xb)                                                # (B, T, head_size=n_embd/n_head)
        q = self.query(xb)                                              # (B, T, head_size)
        v = self.value(xb)                                              # (B, T, head_size)

        B, T, C = k.shape

        weights = q @ k.transpose(-2, -1) / C**0.5                      # (B, T, C) @ (B, C, T) => (B, T, T)
                                                                        # divided by n_embd root to normalize the values, bring them to one

        

        mask = torch.tril(torch.ones(T, T, device=xb.device))          # triangular mask, built on the same device as the input
        weights = qkv.masked_fill(mask == 0, float('-inf'))      
        weights = F.softmax(weights, dim=-1)                            # softmax to get probabilities (B, T, T)
        weights = self.dropout(weights)

        out = weights @ v                                               # (B, T, T) @ (B, T, head_size) => (B, T, head_size)
        return out
      

In [230]:
class MultipleSelfAttention(nn.Module):
    def __init__(self, n_head, n_embd):
        super().__init__()
        self.heads = nn.ModuleList([SelfAttention(n_embd // n_head) for i in range(n_head)])
        self.lin_proj = nn.Linear(n_embd, n_embd)

    def forward(self, xb):
        full_attention = []

        for head in self.heads: 
            full_attention.append(head(xb))                             # (B, T, head_size)
        
        full_attention = torch.cat(full_attention, dim=-1)              # add n_head of (B, T, head_size) along last dimension = (B, T, C)
        full_attention = self.lin_proj(full_attention)                  # linear projection of (B, T, C) to get (B, T, C)

        return full_attention


In [231]:
class CompleteSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.head_size = n_embd // n_head
        # self.key = nn.Linear(n_embd, n_embd)
        # self.query = nn.Linear(n_embd, n_embd)
        # self.value = nn.Linear(n_embd, n_embd)
        self.qkv = nn.Linear(n_embd, n_embd*3)
        self.dropout = nn.Dropout(dropout)
        self.lin_proj = nn.Linear(n_embd, n_embd)
        self.register_buffer('mask', torch.tril(torch.ones(block_size, block_size)))

        


    def forward(self, xb, sdpa=True):
        B, T, C = xb.shape

        qkv = self.qkv(xb)

        q, k, v = qkv.split(n_embd, dim=-1)

        k = k.reshape((B, T, n_head, self.head_size)).transpose(-2, -3)                  # (B, n_head, T, head_size) 
        q = q.reshape((B, T, n_head, self.head_size)).transpose(-2, -3)                                  
        v = v.reshape((B, T, n_head, self.head_size)).transpose(-2, -3)    
        
        if sdpa:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=dropout if self.training else 0.0)
        else:
            weights = q @ k.transpose(-2, -1) / self.head_size**0.5                                                  # (B, n_head, T, head_size) @ (B, n_head, head_size, T) 

            weights = weights.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
            weights = F.softmax(weights, dim=-1)
            weights = self.dropout(weights)

            out = (weights @ v)
        out = out.transpose(-2, -3).reshape((B, T, C))
        out = self.lin_proj(out)

        return out




In [232]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.ff = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.GELU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )
    
    def forward(self, xb):
        return self.ff(xb)

In [233]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        # self.full_attend = MultipleSelfAttention(n_head, n_embd)
        self.comp_full_attend = CompleteSelfAttention(n_embd, n_head)
        self.ff = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, xb):
        attention = xb + self.comp_full_attend(self.ln1(xb))                      # (B, T, C) => 4 * self_attend(B, T, C/4) => (B, T, C)
        feed_forward = attention + self.ff(self.ln2(attention))

        return feed_forward

In [234]:
class LTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)         # a token embedding allows us to represent letters as vectors
        self.pos_embedding = nn.Embedding(block_size, n_embd)           # attention also depends on the location of a token
                                                                        # the token embedding itself will give the same value to a character at the front vs at the end
                                                                        # pos embedding handles this
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for i in range(n_layers)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.last_layer = nn.Linear(n_embd, vocab_size)

    def forward(self, xb, targets):
        B, T = xb.shape

        tok_embd = self.token_embedding(xb)                             # (B, T, C)
        pos_embd = self.pos_embedding(torch.arange(T, device=xb.device))  # (T, C), positions built on the input's device
        x = tok_embd + pos_embd                                       # (B, T, C) + (T, C) => (B, T, C) + (B, T, C)

        out = self.blocks(x)
        logits = self.last_layer(self.ln_f(out))
        
        if targets is None: 
            loss = None
        else:
            flat_logits = logits.view(-1, vocab_size)
            flat_targets = targets.view(-1)
            loss = F.cross_entropy(flat_logits, flat_targets)
        return logits, loss
    
    def generate(self, tokens, idx):

        for i in range(tokens):
            logits, loss = self(idx[:, -block_size:], None)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_tok), dim=1)
        
        return idx


xb, yb = generate_batch()

model = LTransformer().to(device)                                   # move all model parameters onto the active device
logits, loss = model(xb, yb)
print(logits, loss)

tensor([[[-0.0836, -0.3224, -1.1327,  ...,  0.5114, -0.1444,  0.0140],
         [ 0.8276,  0.5180,  0.0113,  ...,  0.5212,  0.8809,  0.6941],
         [ 0.1181,  0.6344, -0.2136,  ..., -0.0887, -0.6872, -0.1365],
         ...,
         [ 1.0056,  0.1192, -0.1030,  ...,  0.7950,  0.3154,  0.4594],
         [ 0.4460,  0.1834, -0.5862,  ...,  1.2007,  0.1133,  0.3785],
         [ 0.7522, -0.1324,  0.9526,  ...,  0.8404, -0.3278, -0.1648]],

        [[ 0.1090, -1.4872, -0.7010,  ..., -0.3042, -0.1675, -0.1221],
         [ 0.4726,  0.1318, -0.6165,  ..., -0.4333, -0.1421,  0.3867],
         [ 0.5841, -0.7010, -0.5295,  ..., -0.6315,  0.5730,  0.5063],
         ...,
         [ 0.9611, -0.2879,  1.0388,  ...,  1.5548, -0.3550,  0.4927],
         [ 1.1086, -0.0856, -0.5597,  ...,  0.5709,  0.1372,  0.1529],
         [ 0.2356,  0.8061, -0.0901,  ...,  0.1082, -1.0161, -0.6464]],

        [[ 0.2450, -0.4766, -0.8741,  ..., -0.2657,  0.5253, -0.3300],
         [ 0.0897, -0.0056, -0.1690,  ..., -0

In [235]:
# RESUME: load the best checkpoint into `model` to continue training from it.
# This runs right before the training loop, so "Run All" resumes from niche_model.pt.
# To train from scratch instead, run the model-build cell above and SKIP this cell.
checkpoint = torch.load('niche_model.pt', map_location=device)
cfg = checkpoint['config']                          # the architecture the weights were saved with

model = LTransformer().to(device)
model.load_state_dict(checkpoint['model_state'])
print(f"resumed from niche_model.pt (val_loss {checkpoint.get('val_loss', float('nan')):.4f})")

resumed from niche_model.pt (val_loss 1.1472)


In [236]:
import os
import time

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss(iters):
    out = {}
    model.eval()
    for split in ['train', 'test']:
        losses = torch.zeros(iters)
        for k in range(iters):
            xb, yb = generate_batch(split)                  # use the split's data (train vs test)
            tlogits, tloss = model(xb, yb)
            losses[k] = tloss.item()
        out[split] = losses.mean()
    model.train()
    return out

def save_checkpoint(path, val_loss):
    checkpoint = {
        'model_state': model.state_dict(),          # the trained weights
        'config': {                                 # so you can rebuild the exact architecture
            'vocab_size': vocab_size,
            'n_embd': n_embd,
            'n_head': n_head,
            'n_layers': n_layers,
            'block_size': block_size,
            'dropout': dropout,
        },
        'stoi': stoi,                               # char↔token maps (derived from the data)
        'itos': itos,
        'val_loss': val_loss,                       # the test loss this checkpoint achieved
    }
    torch.save(checkpoint, path)

# initialize best_val so a kernel restart can't clobber a better checkpoint:
#   - within a session, carry over whatever best we've already seen
#   - on a fresh kernel, seed from the val_loss already saved on disk (not infinity),
#     so the loop only overwrites niche_model.pt when it genuinely beats it
try:
    best_val
except NameError:
    if os.path.exists('niche_model.pt'):
        best_val = torch.load('niche_model.pt', map_location='cpu').get('val_loss', float('inf'))
        print(f"seeded best_val from niche_model.pt on disk: {best_val:.4f}")
    else:
        best_val = float('inf')

start = time.time()
last = start
for i in range(epochs):
    xb, yb = generate_batch()
    with torch.autocast(device_type=device, dtype=torch.bfloat16):
        logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if i % 1000 ==0:
        estimated_loss = estimate_loss(50)
        test_loss = estimated_loss['test'].item()
        now = time.time()

        improved = test_loss < best_val                          # only the best val point gets persisted
        if improved:
            best_val = test_loss
            save_checkpoint('niche_model.pt', best_val)

        marker = '  <- saved new best' if improved else ''
        print(f"loss at epoch {i}: {estimated_loss['train']:.4f}, {test_loss:.4f} "
              f"(best {best_val:.4f}){marker} | "
              f"total {now - start:.1f}s, +{now - last:.1f}s since last")
        last = now

loss at epoch 0: 1.1161, 1.1618 (best 1.1472) | total 6.9s, +6.9s since last
loss at epoch 1000: 1.0922, 1.1452 (best 1.1452)  <- saved new best | total 156.5s, +149.6s since last
loss at epoch 2000: 1.0801, 1.1448 (best 1.1448)  <- saved new best | total 306.8s, +150.3s since last
loss at epoch 3000: 1.0773, 1.1334 (best 1.1334)  <- saved new best | total 455.5s, +148.7s since last
loss at epoch 4000: 1.0676, 1.1298 (best 1.1298)  <- saved new best | total 604.0s, +148.5s since last
loss at epoch 5000: 1.0623, 1.1276 (best 1.1276)  <- saved new best | total 752.8s, +148.8s since last
loss at epoch 6000: 1.0531, 1.1271 (best 1.1271)  <- saved new best | total 901.6s, +148.8s since last
loss at epoch 7000: 1.0506, 1.1219 (best 1.1219)  <- saved new best | total 1051.5s, +149.9s since last
loss at epoch 8000: 1.0397, 1.1209 (best 1.1209)  <- saved new best | total 1202.3s, +150.7s since last
loss at epoch 9000: 1.0355, 1.1175 (best 1.1175)  <- saved new best | total 1353.4s, +151.1s sinc

In [237]:
# the training loop already auto-saves the BEST val checkpoint to niche_model.pt.
# this is a manual snapshot of the CURRENT (last) weights, saved to a separate file
# so it never clobbers the best checkpoint.
save_checkpoint('niche_model_last.pt', best_val)
print('saved current weights to niche_model_last.pt')

saved current weights to niche_model_last.pt


In [240]:
model.eval()
print('model loaded and in eval mode')
idx = torch.zeros((1, 1), dtype=torch.long, device=device)          # seed token, on the active device
out = model.generate(200, idx)
print(decode(out[0].tolist()))
model.train()
print('\nreturning to training mode')

model loaded and in eval mode

on the Sophcer-task out of power: their wisdom sput a space with prey
themselves,—the stupefies "pleasant" and, whether it be in spite of all
things, the little distortions of their values--what follo

returning to training mode
